# 作业 02：把多个输入接入上周的训练模板

作业 01 比较了同一个样本的两种表示：两个标量输入张量，或一个包含两个数的输入张量。本题把多个样本分别放入 x1、x2，练习补全训练代码。

请参考上周模板，让同一个预测函数同时使用 x1 和 x2。所有样本共享 w1、w2、b；每次预测都需要两个输入。下面提供函数和数据，参数创建、损失、梯度与更新由你完成。

本文件可以独立运行，不依赖作业 01 的内核变量。


In [1]:
import torch

def predict(x1, x2, w1, w2, b):
    return w1 * x1 + w2 * x2 + b


## 训练数据：每一行是同一个样本的两个输入和一个目标

下面 x1、x2、y 的 shape 都为 `(6,)`，表示 6 个样本。`predict(x1, x2, ...)` 同时使用两个输入张量，返回 6 个预测值。

| 样本 i | x1[i] | x2[i] | y[i] |
|---|---|---|---|
| 0 | -2 | 1 | -6 |
| 1 | -1 | 0 | -1 |
| 2 | 0 | 2 | -5 |
| 3 | 1 | -1 | 6 |
| 4 | 2 | 3 | -4 |
| 5 | 3 | 1 | 4 |

训练时不要修改输入和目标，也不要直接填写猜测的参数答案。


In [2]:
x1 = torch.tensor([-2., -1., 0., 1., 2., 3.])
x2 = torch.tensor([1., 0., 2., -1., 3., 1.])
y = torch.tensor([-6., -1., -5., 6., -4., 4.])
print('x1:', x1, 'shape:', tuple(x1.shape))
print('x2:', x2, 'shape:', tuple(x2.shape))
print('y: ', y, 'shape:', tuple(y.shape))


x1: tensor([-2., -1.,  0.,  1.,  2.,  3.]) shape: (6,)
x2: tensor([ 1.,  0.,  2., -1.,  3.,  1.]) shape: (6,)
y:  tensor([-6., -1., -5.,  6., -4.,  4.]) shape: (6,)


## 你的训练区：修改上周模板

创建三个浮点标量参数 w1、w2、b，都启用梯度，并一起交给优化器。每轮调用给定的 predict 函数，同时传入 x1、x2，得到 shape 为 `(6,)` 的 y_pred。

使用绝对值损失：先得到 `(6,)` 的逐样本误差，再用 `.mean()` 得到 shape 为 `()` 的平均损失。只对这个平均损失 backward 一次，然后更新全部参数。

可以从三个参数都为 0、2000 轮、学习率 `0.1 / (epoch ** 0.5)` 开始。这里预测公式是线性的，零初始化与上周的平方参数例子不同。

补完后删除 NotImplementedError。


In [3]:
# 创建三个参数，都初始化为0，开启梯度计算
w1 = torch.tensor(0.0, requires_grad=True)
w2 = torch.tensor(0.0, requires_grad=True)
b = torch.tensor(0.0, requires_grad=True)

# 创建 SGD 优化器，把三个参数都交给它管理
optimizer = torch.optim.SGD([w1, w2, b], lr=0.1)

# 训练循环 2000 轮
for epoch in range(1, 2001):
    # 每轮动态调整学习率，越到后面越小
    lr = 0.1 / (epoch ** 0.5)
    for g in optimizer.param_groups:
        g['lr'] = lr
    
    # 清空上一轮的旧梯度
    optimizer.zero_grad()
    
    # 前向传播：同时传入 x1 和 x2，算出 6 个预测值
    y_pred = predict(x1, x2, w1, w2, b)
    
    # 计算损失：每个样本的绝对误差，再求平均
    per_sample_error = torch.abs(y - y_pred)
    loss = per_sample_error.mean()
    
    # 反向传播 + 更新参数
    loss.backward()
    optimizer.step()

# 训练结束后，不计算梯度，看最终效果
with torch.no_grad():
    y_pred_final = predict(x1, x2, w1, w2, b)
    loss_final = torch.abs(y - y_pred_final).mean()

# 打印结果
print('===== 训练后的参数 =====')
print('w1 =', w1.item(), ' shape:', tuple(w1.shape))
print('w2 =', w2.item(), ' shape:', tuple(w2.shape))
print('b  =', b.item(),  ' shape:', tuple(b.shape))

print('\n===== 输入和目标 =====')
print('x1 =', x1, ' shape:', tuple(x1.shape))
print('x2 =', x2, ' shape:', tuple(x2.shape))
print('y  =', y,  ' shape:', tuple(y.shape))

print('\n===== 预测和损失 =====')
print('y_pred =', y_pred_final, ' shape:', tuple(y_pred_final.shape))
print('逐样本损失 =', torch.abs(y - y_pred_final), ' shape:', tuple(torch.abs(y - y_pred_final).shape))
print('平均损失 =', loss_final.item(), ' shape:', tuple(loss_final.shape))


===== 训练后的参数 =====
w1 = 1.999006986618042  shape: ()
w2 = -3.0010085105895996  shape: ()
b  = 0.9993674159049988  shape: ()

===== 输入和目标 =====
x1 = tensor([-2., -1.,  0.,  1.,  2.,  3.])  shape: (6,)
x2 = tensor([ 1.,  0.,  2., -1.,  3.,  1.])  shape: (6,)
y  = tensor([-6., -1., -5.,  6., -4.,  4.])  shape: (6,)

===== 预测和损失 =====
y_pred = tensor([-5.9997, -0.9996, -5.0026,  5.9994, -4.0056,  3.9954])  shape: (6,)
逐样本损失 = tensor([0.0003, 0.0004, 0.0026, 0.0006, 0.0056, 0.0046])  shape: (6,)
平均损失 = 0.0023727715015411377  shape: ()


## 我的观察与实验

1. 现在一次预测需要几个输入张量？x1 和 x2 与样本数 6 分别是什么关系？（10分）
2. 对照上周：预测公式、参数列表和损失计算分别改了哪里？优化器的基本步骤变了吗？（10分）
3. 记录最终参数、预测和平均损失，说明如何判断训练有效。（20分）
4. 为什么这次 w1=w2=b=0 仍然可以开始学习？（10分）

待填写。


In [ ]:


- 一次预测需要 **2 个输入张量**：`x1` 和 `x2`。
- `x1` 和 `x2` 的 shape 都是 `(6,)`，分别表示 6 个样本各自的第一个输入和第二个输入。
- 也就是说：`x1[i]` 是第 i 个样本的第一个输入值，`x2[i]` 是第 i 个样本的第二个输入值，`y[i]` 是第 i 个样本的目标值。一次 predict 调用同时算出全部 6个样本

- **预测公式**：上周是单输入 `y = w*x + b`，这次变成双输入 `y = w1*x1 + w2*x2 + b`，多了一个w2。
- **参数列表**：从上周的 2 个参数（w、b）变成 3 个参数（w1、w2、b）。
- **损失计算**：还是用绝对值损失（每个样本误差取绝对值，再求平均），没有变。
- **优化器的基本步骤**：完全没变，还是那 5 步：清空梯度 → 前向传播算预测 → 算损失 → 反向传播算梯度 → 更新参数。只是多传了一个输入和一个参数。


- 最终 w1 ≈ ___，w2 ≈ ___，b ≈ ___
- 最终平均损失 ≈ ___
- 判断训练有效的依据：**平均损失从初始的较大值（约 3~4）一路下降到一个很小的稳定值**，而且预测值 `y_pred` 和真实值 `y` 很接近，就说明训练有效。如果损失不下降或者震荡，就说明学习率或初始化有问题。

**4. 为什么这次 w1=w2=b=0 仍然可以开始学习？**

- 因为这次的预测公式是**线性模型**：`y = w1*x1 + w2*x2 + b`。
- 当 w1=w2=b=0 时，预测值全是 0，但只要 x1、x2 不全是 0，损失（真实值 y 减去 0）就不为 0，损失对 w1、w2 的梯度也不为 0，所以参数就能正常更新。
- 上周如果是带平方项的模型（比如 y = w*x² + b），当 w=0 时梯度恰好也是 0，就卡住学不动了。这次是纯线性，零初始化不会导致梯度为 0，所以可以正常开始学习